[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/batch/server-side-batching/server_side_batching.ipynb)

# Server-Side Batching

Bulk imports into Weaviate have always been a balancing act. Send batches that are too small and you waste round trips; send batches that are too large and you risk timeouts, memory pressure, or overwhelming a vectorizer. Historically you tuned `batch_size` by hand, per dataset, per cluster.

**Server-side batching (SSB)** moves that decision to where the information actually lives: the server. The client streams objects, and Weaviate tells it how fast to send them, adjusting continuously based on real load. There is no batch size to tune.

This recipe covers:

1. `collection.data.ingest()` with an in-memory list — the one-liner
2. `collection.data.ingest()` with a generator — streaming a large file without loading it into memory
3. `collection.batch.stream()` — the context manager, for per-object control and error monitoring
4. How errors surface in each style
5. An honest comparison against client-side batching, and when each one is the right call

## Requirements

| | Minimum |
|---|---|
| Weaviate | `v1.36` (this recipe's outputs were captured on `v1.39.4`) |
| `weaviate-client` (Python) | `v4.20.0` |

Server-side batching rides on the gRPC API, which current clients enable automatically — there is nothing extra to configure.

> Server-side batching is also available in the TypeScript, Java and C# clients. The Go client does not support it; use manual batching there.

## Setup

This recipe talks to a **local Weaviate** and supplies its own vectors, so it needs no API keys and no inference provider. If you don't have an instance running, start one in a separate terminal:

```bash
docker run -p 8080:8080 -p 50051:50051 cr.weaviate.io/semitechnologies/weaviate:1.39.4
```

Everything below works unchanged against Weaviate Cloud or any other `1.36+` deployment — swap the `connect_to_local()` call for `connect_to_weaviate_cloud()`.

In [ ]:
%pip install -q -U "weaviate-client>=4.20.0"

In [2]:
import weaviate

client = weaviate.connect_to_local()

server_version = client.get_meta()["version"]
print(f"weaviate-client : {weaviate.__version__}")
print(f"Weaviate server : {server_version}")

major, minor, *_ = (int(p) for p in server_version.split("-")[0].split("."))
assert (major, minor) >= (1, 36), "Server-side batching requires Weaviate 1.36 or newer"
print("\nServer-side batching is available.")

weaviate-client : 4.23.1
Weaviate server : 1.39.4

Server-side batching is available.


## The data

We use the 1,000-product sample dataset from this repo. To keep the recipe dependency-free and focused on *batching* rather than on any particular embedding provider, we compute vectors ourselves with a tiny hashing embedder — roughly a bag-of-words projected into a fixed-size vector.

It is not a semantic model, but it is deterministic, instant, and good enough that search results at the end are recognisably relevant. In a real import you would configure a vectorizer on the collection and skip this entirely.

In [3]:
import csv
import hashlib
import io
import math
import urllib.request

DATASET_URL = (
    "https://raw.githubusercontent.com/weaviate/recipes/main/datasets/1k_products.csv"
)

raw = urllib.request.urlopen(DATASET_URL).read().decode()
products = list(csv.DictReader(io.StringIO(raw)))

print(f"{len(products)} products")
print(products[0])

1000 products
{'name': "Buck 110 Folding Hunter's Knife", 'description': "The world's #1-selling hunting knife in its original lock-back design.", 'url': 'https://www.llbean.com/llb/shop/38592?page=buck-110-folding-hunter-s-knife&bc=29-507929&feat=507929-gn0&csp=f'}


In [4]:
VECTOR_DIM = 256


def embed(text: str) -> list[float]:
    """A dependency-free hashing embedder. Stands in for a real vectorizer."""
    vector = [0.0] * VECTOR_DIM
    for token in text.lower().split():
        bucket = int.from_bytes(hashlib.md5(token.encode()).digest()[:8], "big")
        vector[bucket % VECTOR_DIM] += 1.0
    norm = math.sqrt(sum(x * x for x in vector)) or 1.0
    return [x / norm for x in vector]


def product_text(row: dict) -> str:
    return f"{row['name']} {row['description']}"


sample = embed("waterproof hiking boots")
print(f"dimensions       : {len(sample)}")
print(f"non-zero buckets : {sum(1 for x in sample if x)}")
print(f"norm             : {math.sqrt(sum(x * x for x in sample)):.3f}")

dimensions       : 256
non-zero buckets : 3
norm             : 1.000


## Create the collection

`Configure.Vectors.self_provided()` tells Weaviate that we will supply vectors with each object. Swap in `Configure.Vectors.text2vec_weaviate()` (or any other provider) and every import below works identically — you simply stop passing `vector=`.

In [5]:
from weaviate.classes.config import Configure, DataType, Property


def recreate_collection(name: str):
    """Drop and recreate a collection so each section starts from a clean slate."""
    if client.collections.exists(name):
        client.collections.delete(name)
    return client.collections.create(
        name=name,
        properties=[
            Property(name="name", data_type=DataType.TEXT),
            Property(name="description", data_type=DataType.TEXT),
            Property(name="url", data_type=DataType.TEXT, index_searchable=False),
        ],
        vector_config=Configure.Vectors.self_provided(),
    )

## 1. `data.ingest()` with a list

The simplest entry point. Hand `ingest()` any iterable of objects and it imports the lot, internally splitting it into server-directed batches. Unlike `insert_many()`, which sends everything as a single request and will fail if the payload exceeds your instance's maximum, `ingest()` has no size ceiling you need to think about.

The return value is a `BatchObjectReturn`, the same type `insert_many()` gives you.

In [6]:
import time

from weaviate.classes.data import DataObject

products_collection = recreate_collection("SsbProducts")

objects = [
    DataObject(
        properties={
            "name": row["name"],
            "description": row["description"],
            "url": row["url"],
        },
        vector=embed(product_text(row)),
    )
    for row in products
]

start = time.perf_counter()
result = products_collection.data.ingest(objects)
elapsed = time.perf_counter() - start

print(f"imported : {len(result.uuids)} objects in {elapsed:.2f}s")
print(f"errors   : {len(result.errors)}")
print(f"in Weaviate: {products_collection.aggregate.over_all(total_count=True).total_count}")

imported : 1000 objects in 0.17s
errors   : 0
in Weaviate: 1000


## 2. `data.ingest()` with a generator

The list above had to fit in memory. For anything genuinely large, pass a **generator** instead: `ingest()` pulls objects as it goes, so only the objects currently in flight are resident. This is the pattern to reach for when importing a multi-gigabyte file.

Let's build a 20,000-line JSONL file to import from.

In [7]:
import json
import os

JSONL_PATH = "products_20k.jsonl"
N_ROWS = 20_000

with open(JSONL_PATH, "w") as f:
    for i in range(N_ROWS):
        row = products[i % len(products)]
        f.write(
            json.dumps(
                {
                    "name": f"{row['name']} #{i}",
                    "description": row["description"],
                    "url": row["url"],
                }
            )
            + "\n"
        )

print(f"{JSONL_PATH}: {N_ROWS:,} rows, {os.path.getsize(JSONL_PATH) / 1e6:.1f} MB")

products_20k.jsonl: 20,000 rows, 7.4 MB


In [8]:
def read_objects(path: str):
    """Yield one DataObject per line. Nothing is held in memory but the current row."""
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            yield DataObject(properties=record, vector=embed(product_text(record)))


stream_collection = recreate_collection("SsbStream")

start = time.perf_counter()
result = stream_collection.data.ingest(read_objects(JSONL_PATH))
elapsed = time.perf_counter() - start

print(f"imported : {len(result.uuids):,} objects in {elapsed:.2f}s")
print(f"errors   : {len(result.errors)}")
print(f"in Weaviate: {stream_collection.aggregate.over_all(total_count=True).total_count:,}")

imported : 20,000 objects in 3.99s
errors   : 0
in Weaviate: 20,000


## 3. `batch.stream()` — the context manager

If you're coming from `batch.fixed_size()` or `batch.dynamic()`, this will look familiar. `batch.stream()` is the same context-manager shape, but the send rate is dictated by the server rather than by a `batch_size` you picked.

It gives you two things `ingest()` doesn't:

- **`batch.number_errors`** — a running error count you can inspect mid-import, so a broken dataset doesn't have to run to completion before you find out.
- **`concurrency`** — how many batch requests are in flight at once (default `1`). Worth raising when the client is waiting on network latency rather than on the server.

The final batch is flushed automatically when the block exits.

In [9]:
ctx_collection = recreate_collection("SsbContext")

start = time.perf_counter()
with ctx_collection.batch.stream(concurrency=4) as batch, open(JSONL_PATH) as f:
    for line in f:
        record = json.loads(line)
        batch.add_object(properties=record, vector=embed(product_text(record)))

        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break
elapsed = time.perf_counter() - start

failed = ctx_collection.batch.failed_objects
print(f"imported : {elapsed:.2f}s")
print(f"failed   : {len(failed)}")
print(f"in Weaviate: {ctx_collection.aggregate.over_all(total_count=True).total_count:,}")

imported : 3.96s
failed   : 0
in Weaviate: 20,000


## Errors

Server-side batching never raises on a bad object — a single malformed row shouldn't abort a ten-hour import. Failures are collected and handed back instead, and where you read them depends on the entry point:

- `data.ingest()` → the returned `BatchObjectReturn.errors`, a dict keyed by the object's index in your input
- `batch.stream()` → `collection.batch.failed_objects` after the block exits

Each error carries the server's `message` and the original `object_`, so you can fix and retry just the failures.

In [10]:
error_collection = recreate_collection("SsbErrors")

mixed = [
    DataObject(
        properties={"name": "Trail Model Boots", "description": "Waterproof.", "url": "https://example.com/1"},
        vector=embed("Trail Model Boots Waterproof"),
    ),
    # `name` is declared TEXT, but this row supplies a list -> the server rejects it.
    DataObject(
        properties={"name": ["not", "a", "string"], "description": "Broken row.", "url": "https://example.com/2"},
        vector=embed("broken row"),
    ),
]

result = error_collection.data.ingest(mixed)

print(f"has_errors : {result.has_errors}")
print(f"succeeded  : {len(result.uuids)}")
print(f"failed     : {len(result.errors)}\n")

for index, error in result.errors.items():
    print(f"index {index}: {error.message}")

{'error': "invalid text property 'name' on class 'SsbErrors': not a string, but []interface {}", 'object': '911ec653-af74-439d-9ccb-d3c04badb310', 'action': 'use {client,collection}.batch.failed_objects to access this error'}


has_errors : True
succeeded  : 1
failed     : 1

index 1: invalid text property 'name' on class 'SsbErrors': not a string, but []interface {}


## How does it compare to client-side batching?

Worth measuring rather than assuming. Below we import the same 20,000 objects five ways, with the vectors pre-computed so we're timing the import and not the embedder.

In [11]:
with open(JSONL_PATH) as f:
    rows = [json.loads(line) for line in f]

prepared = [(row, embed(product_text(row))) for row in rows]

print(f"{len(prepared):,} objects prepared\n")


def time_import(label, batching_context):
    collection = recreate_collection("SsbBenchmark")
    start = time.perf_counter()
    with batching_context(collection) as batch:
        for properties, vector in prepared:
            batch.add_object(properties=properties, vector=vector)
    elapsed = time.perf_counter() - start
    count = collection.aggregate.over_all(total_count=True).total_count
    print(f"{label:<26} {elapsed:6.2f}s   {count:,} objects")


time_import("fixed_size(batch_size=100)", lambda c: c.batch.fixed_size(batch_size=100))
time_import("fixed_size(batch_size=1000)", lambda c: c.batch.fixed_size(batch_size=1000))
time_import("dynamic()", lambda c: c.batch.dynamic())
time_import("stream()", lambda c: c.batch.stream())
time_import("stream(concurrency=4)", lambda c: c.batch.stream(concurrency=4))

benchmark_collection = recreate_collection("SsbBenchmark")
start = time.perf_counter()
benchmark_collection.data.ingest(
    [DataObject(properties=p, vector=v) for p, v in prepared]
)
elapsed = time.perf_counter() - start
print(f"{'data.ingest(list)':<26} {elapsed:6.2f}s   "
      f"{benchmark_collection.aggregate.over_all(total_count=True).total_count:,} objects")

20,000 objects prepared



fixed_size(batch_size=100)   3.17s   20,000 objects


fixed_size(batch_size=1000)   2.27s   20,000 objects


dynamic()                    4.35s   20,000 objects


stream()                     3.46s   20,000 objects


stream(concurrency=4)        3.24s   20,000 objects


data.ingest(list)            3.36s   20,000 objects


### Reading those numbers honestly

Against an idle local instance with pre-computed vectors, all six land in roughly the same band — and a hand-picked `fixed_size` may well come out on top. That is the expected result, and it's worth being clear about why: on localhost there is no network latency to hide and no vectorizer applying backpressure, so there is very little for the server to adapt *to*. A fixed batch size tuned for exactly these conditions is hard to beat in exactly these conditions.

The case for server-side batching is what happens when conditions stop being ideal:

- **The server sets the rate.** When a vectorizer is inline, or the node is compacting, or another tenant is hammering the same cluster, the server throttles the stream instead of accepting work it can't keep up with. A fixed batch size doesn't know any of that happened.
- **Nothing to tune.** `batch_size=1000` being the winner here says nothing about the next dataset, object shape, or cluster. That tuning pass just stops existing.
- **Load-shedding beats timeouts.** Backpressure degrades gracefully; oversized batches against a busy server fail hard.
- **No payload ceiling.** Unlike `insert_many()`, `ingest()` splits the work for you, so a large input can't blow the maximum request size.

Put bluntly: you adopt SSB to stop thinking about batch sizes and to stay well-behaved under real load, not to win a benchmark on an idle laptop.

## Verify the import

Vectors went in with the objects, so we can query with the same hashing embedder.

In [12]:
response = products_collection.query.near_vector(
    near_vector=embed("waterproof hiking boots"),
    limit=5,
    return_properties=["name"],
)

for obj in response.objects:
    print(obj.properties["name"])

Men's Trail Model 4 Waterproof Hiking Boots
Men's Merrell Moab 2 Waterproof Hiking Boots
Women's Keen Targhee II Waterproof Hiking Boots
Men's Trail Model 4 Waterproof Hiking Boots, Leather/Suede
Women's Vasque Talus Trek Waterproof Hiking Boots


## Which one should you use?

| Situation | Use |
|---|---|
| A list already in memory | `collection.data.ingest(objects)` |
| A large file, stream or cursor | `collection.data.ingest(generator)` |
| You need mid-import error checks, references, or tuned concurrency | `with collection.batch.stream() as batch:` |
| A handful of objects, one request, and you want it to fail loudly | `collection.data.insert_many(objects)` |
| Weaviate `< 1.36`, or the Go client | `collection.batch.fixed_size(...)` |

## Cleanup

In [13]:
for name in ["SsbProducts", "SsbStream", "SsbContext", "SsbErrors", "SsbBenchmark"]:
    if client.collections.exists(name):
        client.collections.delete(name)

if os.path.exists(JSONL_PATH):
    os.remove(JSONL_PATH)

client.close()
print("Done.")

Done.
